# Resume match ranking

Ranks jobs currently in `job_tracker.db` by fit against `resume.md`, using Sentence-Transformer embedding cosine similarity.

**Read-only**: this notebook only reads from `job_tracker.db` and writes a ranked CSV alongside it -- it never writes back into the `jobs` table. Workflow edits (`applied`/`status`/`notes`) stay in `job_viewer_sql.py`'s edit grid, per the project's existing read/write split.

**Chunking, not one big embedding**: The resume is split into sections on markdown `###` headings, each section is embedded separately, and a job's fit score is the **max** similarity across sections -- i.e. "does *any* part of the resume strongly match this job", not "does the resume on average match". Job descriptions are embedded whole (not chunked) -- the relevant requirements/responsibilities text is usually near the top, so tail-truncation risk is lower there.

**This produces a ranking, not a validated classifier.** There's no hand-labeled "good fit" ground truth to score precision/recall against -- treat `fit_score` as *relative* ordering signal within this run, not an absolute, calibrated probability.

In [1]:
import re
import sqlite3

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, util

In [2]:
# Config

DB_PATH = "job_tracker.db"   # run from repo root, per CLAUDE.md convention
RESUME_PATH = "resume.md"    # update to wherever your resume.md actually lives
MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"

# 8192-token context (vs. MiniLM's ~256) -- real job descriptions in this DB run a median
# ~676 tokens, mean ~703, with 93.8% of 306 checked rows exceeding MiniLM's limit (measured
# 2026-09-21). MiniLM was silently truncating most job descriptions to roughly their first third.
QUERY_PREFIX = "search_query: "        # prefix for resume chunks (what we're matching FROM)
DOCUMENT_PREFIX = "search_document: "  # prefix for each job posting (what we're matching AGAINST)

EXCLUDE_EXPIRED = True   # drop is_expired = 1
EXCLUDE_APPLIED = True   # drop applied = 1 -- ranking is for deciding what to apply to next
TOP_N = 25

OUTPUT_CSV = "resume_match_ranking.csv"

## Load and section the resume

In [3]:
def clean_markdown(text: str) -> str:
    """Light markdown-syntax strip so embeddings see prose, not markup noise."""
    text = re.sub(r"```.*?```", " ", text, flags=re.DOTALL)  # code blocks
    text = re.sub(r"!\[.*?\]\(.*?\)", " ", text)              # images
    text = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", text)           # links -> link text
    text = re.sub(r"[#*`_>]", " ", text)                       # heading/emphasis/quote markers
    text = re.sub(r"-{3,}", " ", text)                         # horizontal rules
    text = re.sub(r"\s+", " ", text).strip()
    return text


def split_into_items(section_text: str) -> list[str]:
    """Split a section's raw text on top-level numbered list items ('1. ...',
    '2. ...' at the start of a line, not indented) -- e.g. a PROJECTS section
    listing several distinct projects. Sections with 0 or 1 such markers are
    returned as a single chunk (nothing to split)."""
    parts = re.split(r"^\d+\.\s+", section_text, flags=re.MULTILINE)
    parts = [p for p in parts if p.strip()]
    return parts if len(parts) > 1 else [section_text]


def item_label(item_text: str, fallback: str) -> str:
    """First line of a numbered item, cleaned, as a human-readable chunk label
    (e.g. 'Personal Project - Advanced Academic RAG API') -- falls back to a
    generic label only if the first line is empty after cleaning."""
    first_line = item_text.strip().split("\n", 1)[0]
    label = clean_markdown(first_line).strip()
    return label if label else fallback


def split_resume_into_sections(raw_text: str) -> list[tuple[str, str]]:
    """Split on markdown headings of any level from ## to ###### (a resume's
    real section headers won't necessarily be level-2 -- e.g. a top-level '##'
    document title followed by '###' section headers, as in this project's own
    resume examples.md). Anything before the first heading becomes an 'Intro'
    chunk. A section containing several top-level numbered items (e.g. PROJECTS
    listing 3 distinct projects spanning different skill domains) is further
    split per item rather than embedded as one blended chunk -- otherwise an
    LLM/RAG project and a data-engineering ETL project sitting in the same
    section would dilute each other's signal in the section-level embedding,
    which is exactly the dilution problem section-chunking exists to avoid."""
    parts = re.split(r"^#{2,6}\s+(.+)$", raw_text, flags=re.MULTILINE)
    sections = []
    if parts[0].strip():
        cleaned = clean_markdown(parts[0])
        if cleaned:
            sections.append(("Intro", cleaned))
    for heading, body in zip(parts[1::2], parts[2::2]):
        heading = heading.strip()
        items = split_into_items(body)
        if len(items) == 1:
            cleaned = clean_markdown(items[0])
            if cleaned:
                sections.append((heading, cleaned))
        else:
            for i, item in enumerate(items, start=1):
                cleaned = clean_markdown(item)
                if cleaned:
                    sections.append((item_label(item, f"{heading} #{i}"), cleaned))
    return sections

with open(RESUME_PATH, "r", encoding="utf-8") as f:
    resume_raw = f.read()

resume_sections = split_resume_into_sections(resume_raw)
section_names = [name for name, _ in resume_sections]
section_texts = [text for _, text in resume_sections]

print(f"Resume sections found: {section_names}")

Resume sections found: ['Research Engineer, Intern (A STAR I2R, 6 months)', 'Data Scientist, Intern (SATS, 3 months)', 'Personal Project – Multi-Source Job Aggregation & Tracking Pipeline', 'Personal Project – Advanced Academic RAG API', 'Final Year Project – Speech Emotion Recognition', 'Personal Project – Automated ETL Pipeline (NASA API)', 'Machine Learning Project – Horse Health Outcome Prediction', 'Neural Network & Deep Learning Project – Question Classification', 'Data Mining Project – California Weather Clustering', 'Software Engineering Project – Routing Application', 'Data Science Project - Box Office Revenue Prediction', 'COMPETITIONS', 'Technical Skills']


## Load candidate jobs from the tracker

In [4]:
conn = sqlite3.connect(DB_PATH)

where_clauses = []
if EXCLUDE_EXPIRED:
    where_clauses.append("(is_expired IS NULL OR is_expired = 0)")
if EXCLUDE_APPLIED:
    where_clauses.append("(applied IS NULL OR applied = 0)")
where_sql = f"WHERE {' AND '.join(where_clauses)}" if where_clauses else ""

query = f"""
    SELECT job_id, title, company, source, location, salary_str, work_arrangement,
           seniority, visa_eligibility, min_years_exp, job_url,
           is_agent, is_ai_llm, is_de, is_ds, is_swe,
           description
    FROM jobs
    {where_sql}
"""
jobs_df = pd.read_sql_query(query, conn)
conn.close()

jobs_df = jobs_df.dropna(subset=["description"]).reset_index(drop=True)
print(f"Candidate jobs: {len(jobs_df)}")

Candidate jobs: 305


## Embed and score

`fit_score` = max cosine similarity between the job's (title + description) embedding and any single resume-section embedding. `best_matching_section` records which section drove the score, for sanity-checking the ranking rather than trusting a bare number.

In [ ]:
model = SentenceTransformer(MODEL_NAME, trust_remote_code=True)

section_texts_prefixed = [QUERY_PREFIX + t for t in section_texts]
section_embeddings = model.encode(section_texts_prefixed, convert_to_tensor=True)

job_texts = (jobs_df["title"].fillna("") + ". " + jobs_df["description"].fillna("")).tolist()
job_texts_prefixed = [DOCUMENT_PREFIX + t for t in job_texts]
job_embeddings = model.encode(job_texts_prefixed, convert_to_tensor=True, show_progress_bar=True)

similarity_matrix = util.cos_sim(job_embeddings, section_embeddings).cpu().numpy()  # (n_jobs, n_sections)

jobs_df["fit_score"] = similarity_matrix.max(axis=1)
jobs_df["best_matching_section"] = [section_names[i] for i in similarity_matrix.argmax(axis=1)]

## Ranked shortlist

In [ ]:
display_cols = [
    "job_id", "title", "company", "source", "fit_score", "best_matching_section",
    "salary_str", "work_arrangement", "seniority", "visa_eligibility",
    "is_agent", "is_ai_llm", "is_de", "is_ds", "is_swe", "job_url",
]

ranked = jobs_df.sort_values("fit_score", ascending=False).reset_index(drop=True)

In [ ]:
ranked[display_cols].to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(ranked)} ranked rows to {OUTPUT_CSV}")

## Notes / caveats

- `fit_score` is uncalibrated -- there's no labeled "actually a good fit" ground truth to validate against, so don't read e.g. 0.55 as "55% match"; only the *relative order* within a single run is meaningful, and only among jobs embedded together in that run.
- Section-level max-pooling means a resume with one very on-topic section (e.g. a matching side project) can outrank a resume that's evenly-but-weakly relevant everywhere. That's the intended behavior here, but worth knowing if a ranking looks surprising.
- `job_url` isn't validated for staleness beyond the `is_expired` filter -- `CLAUDE.md` notes `is_expired` is manual-edit-only and doesn't get auto-refreshed on re-scrape, so a listing that expired since it was last scraped may still show up as a top match.
- This is a separate, lighter-weight tool from the `apply-from-tracker`/`cover-letter-generator` skills, which do LLM-based fit reasoning + project ranking for one job at a time. Use this notebook to narrow ~hundreds of tracked jobs down to a shortlist, then run those skills on the specific `job_id`s worth a closer look.